# bioseq bench `report.json` viewer

Self-contained, single-file notebook for inspecting a `report.json`
produced by `./build.sh bench`. The bench writes the file to
`build/report.json` (or whatever path was passed via the
`BIOSEQ_BENCH_REPORT_PATH` environment variable). Override the
`REPORT_PATH` constant in the next cell to point somewhere else.

No external dependencies beyond `plotly` (already available on this
machine) and the Python standard library. To run:

1. Open in Jupyter (`jupyter notebook view_report.ipynb`) or VS Code.
2. Execute the cells top-to-bottom. The last cell renders every
   chart in the report.

Schema: `bioseq-bench-report-v1` — see
`main/bioseq/RESEARCH/BENCHMARKING_SPEEDUP/REPORT_PLOTLY_JSON_DATA.md`
for what each chart means.

In [1]:
!pip3 install plotly nbformat


[notice] A new release of pip is available: 25.1.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [2]:
# ----- configuration -----
# Either an absolute path or a path relative to the notebook directory.
# The bench writes to ./report.json by default; the build script invokes
# the binary from main/build/, so the canonical location is
# main/build/report.json.
REPORT_PATH = '../../../build/report.json'

import json
import os
import sys

import plotly.io
from IPython.display import display, Markdown

with open(REPORT_PATH, 'r') as report_file:
    report = json.load(report_file)

expected_schema = 'bioseq-bench-report-v1'
actual_schema = report.get('schema', '<missing>')
if actual_schema != expected_schema:
    print(
        f'WARNING: schema mismatch — expected {expected_schema!r}, '
        f'got {actual_schema!r}. The notebook may render incorrectly.',
        file=sys.stderr,
    )

print(f'Loaded {os.path.abspath(REPORT_PATH)}')
print(f'  schema:           {actual_schema}')
print(f'  sample_count:     {report.get("sample_count", "?")}')
print(f'  algorithm_count:  {report.get("algorithm_count", "?")}')
print(f'  chart_count:      {len(report.get("charts", {}))}')

Loaded /Users/styczynski/Code/bioseqdb/main/build/report.json
  schema:           bioseq-bench-report-v1
  sample_count:     1008
  algorithm_count:  14
  chart_count:      17


In [3]:
# Render every chart, in the order they appear in the report.
# Each chart prints its short_description as a heading and its
# long_description as a paragraph below the figure. Figures use
# Plotly's interactive renderer — hover for tooltips, drag to zoom,
# double-click to reset.

for chart_id, chart in report.get('charts', {}).items():
    short_description = chart.get('short_description', chart_id)
    long_description = chart.get('long_description', '')
    plotly_payload = chart.get('plotly')
    if plotly_payload is None:
        display(Markdown(f'### `{chart_id}` — _no plotly payload, skipped_'))
        continue
    display(Markdown(f'## `{chart_id}` — {short_description}'))
    try:
        figure = plotly.io.from_json(json.dumps(plotly_payload))
    except Exception as exc:  # noqa: BLE001 — surface any parse error inline
        display(Markdown(
            f'**plotly.io.from_json failed:** `{exc}`'))
        continue
    figure.show()
    if long_description:
        display(Markdown(f'> {long_description}'))

## `seq_length_vs_speed_mean` — Mean alignment latency vs. effective query length

> Mean wall-clock latency per (query, target) pair as a function of the effective query length (post-error length, in bp). One trace per algorithm configuration; both axes log-scale because the length axis spans short reads (300 bp) through assembly fragments (2 Mbp). Lower lines = faster algorithms. Use this chart to spot algorithmic complexity: linear-in-length lines indicate O(n) behaviour, super-linear curves point at O(n log n) or worse regions in the chosen pipeline.

## `seq_length_vs_speed_p99` — Pooled P50 and P99 latency per (algorithm, length bucket)

> Per-bucket pooled P50 and P99 latencies. For each algorithm and length bucket we pool every per-sample timing across all pairs in that bucket (16 pairs × 8 samples = 128 samples for SHORT, 24 for MEDIUM, 12 for LONG, 1 for VERY_LONG) before computing percentiles, so the P99 is over a sufficient sample size (except VERY_LONG, where it degenerates to the single observation). Solid markers = P50, hollow markers = P99. The vertical gap between the two for one algorithm is the tail width: large gaps indicate a heavy-tailed latency distribution and small gaps indicate a tight distribution.

## `seq_length_vs_memory` — Working-set delta (current RSS) vs. effective query length

...

> Heatmap rows are algorithm configurations; columns are the 12 (length-bucket, error-rate) cells in the sweep matrix. Cell colour encodes the position-accuracy fraction in that cell. Unlike the algorithm × length-only heatmap, this view preserves the error-rate axis — a configuration that's perfect on LOW-error but collapses on HIGH-error appears as two distinct columns rather than averaging to 50%.

## `mapq_calibration` — MAPQ calibration curve (accuracy by MAPQ bin)

> For each algorithm we bin primaries into 10-wide MAPQ bins (0-9, 10-19, ..., 50-60) and report the fraction correctly placed in each bin. A well-calibrated aligner produces a monotonic curve climbing from low accuracy at low MAPQ to ~1.0 at MAPQ=60 (since MAPQ is Phred-scaled and 60 means 1-in-1,000,000 expected misplacement). Curves that cluster at high accuracy across all MAPQs indicate MAPQ deflation (the aligner is more confident than its scores reflect); curves with low accuracy at high MAPQ indicate inflation.

## `algorithm_summary_table` — Per-algorithm summary table

> Tabular roll-up of every algorithm's pair count, mean / median / P99 latency in milliseconds, and overall position-accuracy fraction. Useful as a one-glance sanity check before diving into the per-chart detail; the row ordering matches the chart legend order so the reader can cross-reference.